In [ ]:
import os
import scvi
import torch
import pandas as pd
import scanpy as sc

torch.set_num_threads(20) # limit the number of threads used (otherwise it hogs all available threads)
!OMP_NUM_THREADS=20

pd.set_option('display.max_columns', None)

os.chdir("scRNA-seq")

ImportError: cannot import name 'safe_zip' from 'jax.util' (/mnt/lareaulab/reliscu/anaconda3/envs/splicing_python_env/lib/python3.11/site-packages/jax/util.py)

In [ ]:
TMPDIR=/mnt/lareaulab/reliscu/tmp pip install scvi-tools==1.1.2 --cache-dir /mnt/lareaulab/reliscu/tmp # done
TMPDIR=/mnt/lareaulab/reliscu/tmp pip install scanpy==1.10.1 --cache-dir /mnt/lareaulab/reliscu/tmp # done
TMPDIR=/mnt/lareaulab/reliscu/tmp pip3 install igraph==0.11.6 --cache-dir /mnt/lareaulab/reliscu/tmp # done

In [ ]:
# note: Krizay 2022 data will be made publicly available after publication

adata = sc.read_mtx("data/krizay_2022/matrix.mtx") 
adata = adata.T
genes = pd.read_csv("data/krizay_2022/genes.tsv", sep="\t", header=None)
adata.var_names = [line.strip().split("\t")[0] for line in open("data/krizay_2022/genes.tsv")]
adata.obs_names = [line.strip() for line in open("data/krizay_2022/barcodes.tsv")]
adata.var["gene_names"] = genes[1].values

In [23]:
# Filter lowly expressed genes
sc.pp.filter_genes(adata, min_cells = 3)

In [ ]:
adata.layers["counts"] = adata.X.copy() # store the raw counts

# subset count data to highly variable features
sc.pp.highly_variable_genes(adata, layer="counts", flavor="seurat_v3", n_top_genes=2000)
adata = adata[:, adata.var.highly_variable]
adata.X = adata.layers["counts"].copy()
adata = adata.copy()


In [ ]:
# run scvi

scvi.model.SCVI.setup_anndata(adata, layer="counts")
model = scvi.model.SCVI(adata)
model.train()
model.save("data/scVI", overwrite=True)

In [ ]:
# extract low dim rep. and save
latent = model.get_latent_representation()
adata.obsm["scVI"] = latent
adata.write('data/scVI/matrix_scvi.h5ad')